# 06 · Running the pipeline in the cloud

The 1,066-star training run took about **11 hours** on my laptop, one star at a time, and stopped whenever the laptop slept. This notebook covers the same run on **AWS**: how it was set up, how long it took and why, whether the cloud gives the same answers, and how its results became a SQL-queryable **data lake**.

## The architecture

| Piece | AWS service | Role |
|---|---|---|
| Container image | **ECR** | The pipeline packaged with Docker (`exoplanet-pipeline:v2`), so it runs identically anywhere |
| Storage | **S3** | Target lists in, results out, and the data lake |
| Compute | **AWS Batch** on **Fargate Spot** | Runs many copies of the container in parallel on discounted spare capacity |
| Permissions | **IAM** | An execution role (pull the image, write logs) and a job role limited to one bucket (least privilege) |
| Logs | **CloudWatch Logs** | Each container's output, kept for 14 days |
| Queries | **Athena** | SQL directly on the Parquet files in S3 |

The run was an **array job**: 32 copies of the container, each processing its own chunk of 34 stars (`run.py --chunk-size 34`), reading the target list from S3 and uploading its results to S3 when done. The JSON settings for the roles, compute environment, and job definition are in the `infra/` folder.

## The benchmark: 4 hours 33 minutes

All **32 chunks succeeded**, in **4.5 hours vs 11 locally** (~2.4× faster), for roughly **$0.30–0.40** of Fargate Spot capacity. Less than the arithmetic promised, and the per-chunk timeline showed why:

1. **Account quota.** After an initial burst of 10, only ~4 containers ran at once: my new account's Fargate Spot limit of 4 vCPUs. On average, 4.5 ran concurrently.
2. **Load imbalance.** The target list puts all planets first. Confirmed planets have many sectors (long baselines, large search grids), so chunks 0 to 12 took 40 to 106 minutes while later chunks took 10 to 30.
3. **Slower CPUs.** The run used about 20 container-hours for 11 laptop-hours of work: each cloud vCPU (typically half a physical core) was ~1.8× slower per star.

**Fixes for next time:** request a higher Fargate Spot quota (the main limit), and shuffle targets before chunking. With 32 vCPUs, the same work would take under an hour.

## Getting the results

The results were downloaded from S3 (plots skipped) with:

```powershell
aws s3 sync s3://exoplanet-pipeline-emaad/results/train_v1_cloud data\train_v1_cloud --exclude "*/plots/*"
```

## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*oktopus.*")

import sys
sys.path.append("..")
import pandas as pd
from src.batch import DATA_DIR, classify_match

---
# Part 1: Combine the 32 chunks

Each chunk saved its own results file. Stack them into one table:

In [2]:
chunks = sorted((DATA_DIR / "train_v1_cloud").glob("chunk_*/results.parquet"))
if not chunks:
    raise FileNotFoundError("No cloud results found: run the aws s3 sync command above first.")

cloud = pd.concat([pd.read_parquet(p) for p in chunks], ignore_index=True)
print(len(chunks), "chunks,", len(cloud), "rows")
cloud["status"].value_counts()

32 chunks, 1066 rows


status
ok       937
error    129
Name: count, dtype: int64

The cloud run had **no network failures**: the stars that failed on the laptop with connection errors or timeouts all succeeded here. The remaining errors are the genuine "no 2-minute data" cases.

---
# Part 2: Does the cloud give the same answers?

Same code, same pinned library versions, so the results *should* match the laptop run. Compare every star that succeeded in both:

In [3]:
local = pd.read_csv(DATA_DIR / "train_v1" / "results.csv")
both = local.merge(cloud, on="target", suffixes=("_local", "_cloud"))
ok_both = both[(both["status_local"] == "ok") & (both["status_cloud"] == "ok")].copy()

# "Identical" = agreeing to within a millionth (exact equality fails on tiny rounding differences)
ok_both["same_period"] = (ok_both["period_days_local"] - ok_both["period_days_cloud"]).abs() < 1e-6
ok_both["same_snr"] = (ok_both["snr_local"] - ok_both["snr_cloud"]).abs() < 1e-6
ok_both["same_sectors"] = ok_both["sectors_local"] == ok_both["sectors_cloud"]

print(f"{len(ok_both)} stars succeeded in both runs")
print(f"identical period: {ok_both['same_period'].sum()}")
print(f"identical SNR: {ok_both['same_snr'].sum()}")

933 stars succeeded in both runs
identical period: 786
identical SNR: 471


Not identical. **First suspect: new data.** TESS releases a new sector roughly monthly, so the two runs (days apart) could have chosen different sectors for some stars:

In [4]:
print(ok_both["same_sectors"].value_counts(), "\n")
pd.crosstab(ok_both["same_sectors"], [ok_both["same_period"], ok_both["same_snr"]])

same_sectors
True     915
False     18
Name: count, dtype: int64 



same_period  False       True       
same_snr     False True  False True 
same_sectors                        
False           16     0     2     0
True           117    14   327   457

Changed sectors explain only **18 stars**. Most differences happened with **identical sectors**, so something else is going on.

**How big are the differences?** The size tells the causes apart: different input data would shift values noticeably across the board, while numerical differences between Windows and Linux would be microscopic.

In [5]:
same_sec = ok_both[ok_both["same_sectors"]].copy()
same_sec["period_rel_diff"] = (same_sec["period_days_cloud"] / same_sec["period_days_local"] - 1).abs()
same_sec["snr_rel_diff"] = (same_sec["snr_cloud"] / same_sec["snr_local"] - 1).abs()
same_sec[["period_rel_diff", "snr_rel_diff"]].describe(percentiles=[0.5, 0.9, 0.99])

,period_rel_diff,snr_rel_diff
count,915.000000,915.000000
mean,0.000778,0.001239
std,0.022484,0.015654
min,0.000000,0.000000
50%,0.000000,0.000000
90%,0.000001,0.000187
99%,0.000069,0.023943
max,0.679819,0.451189


Almost all differences are **microscopic**. For 90% of stars, the period differs by about 0.000001 relative: roughly **one step of the fine period grid**. The two runs picked *neighbouring* grid points, where two adjacent periods fit almost equally well and a tiny floating-point difference between operating systems tips the choice. A handful of weak signals had two separate peaks nearly tied, and the tiny difference flipped which won.

**The question that matters: does it change any conclusion?**

In [6]:
ok_both["match_local"] = [classify_match(p, c) for p, c in zip(ok_both["period_days_local"], ok_both["catalog_period_local"])]
ok_both["match_cloud"] = [classify_match(p, c) for p, c in zip(ok_both["period_days_cloud"], ok_both["catalog_period_cloud"])]

print("Same verdict:", (ok_both["match_local"] == ok_both["match_cloud"]).sum(), "of", len(ok_both))
pd.crosstab(ok_both["match_local"], ok_both["match_cloud"])

Same verdict: 933 of 933


match_cloud,alias,match,miss
match_local,,,
alias,45,0,0
match,0,771,0
miss,0,0,117


**Every star got the same verdict** (match, alias, or miss) in both runs.

So the pipeline's **conclusions are fully reproducible**, and its exact numbers agree to within tiny tolerances for 99% of stars. **Bit-for-bit** reproducibility across operating systems is genuinely hard, which is why teams run everything in one container. From here on, the **cloud results are the canonical dataset**: one consistent, versioned environment.

A side insight: stars whose period *flipped* between runs are inherently ambiguous detections, a possible future reliability flag.

---
# Part 3: Building the data lake

**Athena** runs SQL directly on files in S3, with no database to load. Two things in the raw results would trip it up:

1. **Mixed files in each chunk folder** (CSV, Parquet, plots). Athena needs a folder of only one data format.
2. **Schema drift.** Each chunk was saved separately, so pandas guessed column types per chunk. A column like `error` might be stored as *numbers* in a chunk with no errors (all empty) and as *text* in another. Athena needs one type per column.

The fix: give every column one consistent type, and save one clean Parquet file.

In [7]:
lake = cloud.copy()

# Fix the true/false column first (before the loop below turns it into text)
edge_map = {True: True, False: False, "True": True, "False": False}
lake["at_search_edge"] = lake["at_search_edge"].map(edge_map).astype("boolean")

# Every general "object" column becomes a proper text column, with blanks as missing values
for col in lake.columns:
    if lake[col].dtype == object:
        lake[col] = lake[col].astype("string")

lake.to_parquet(DATA_DIR / "train_v1_lake.parquet", index=False)
lake.dtypes.value_counts()

float64           38
string[python]     5
boolean            1
Name: count, dtype: int64

Uploaded into a **partitioned** layout, where the folder name becomes a column:

```powershell
aws s3 cp data\train_v1_lake.parquet s3://exoplanet-pipeline-emaad/lake/pipeline_results/run=train_v1/train_v1.parquet
```

The `run=train_v1` folder makes `run` a queryable column, so future runs (like `run=candidates_v1`) can sit in the same table.

## The table definition

Athena needs each column's name and type. Rather than typing 44 by hand, I generated the SQL from the table:

In [8]:
type_map = {"float64": "double", "int64": "bigint", "string": "string", "boolean": "boolean"}
cols = ",\n  ".join(f"`{c}` {type_map[str(t).split('[')[0]]}" for c, t in lake.dtypes.items())

ddl = f"""CREATE EXTERNAL TABLE exoplanet.pipeline_results (
  {cols}
)
PARTITIONED BY (run string)
STORED AS PARQUET
LOCATION 's3://exoplanet-pipeline-emaad/lake/pipeline_results/'"""
print(ddl)

CREATE EXTERNAL TABLE exoplanet.pipeline_results (
  `target` string,
  `toi` double,
  `label` string,
  `catalog_period` double,
  `catalog_depth_ppm` double,
  `status` string,
  `error` string,
  `runtime_s` double,
  `tic_id` double,
  `n_sectors` double,
  `sectors` string,
  `n_points` double,
  `baseline_days` double,
  `pmax_searched` double,
  `period_days` double,
  `t0_btjd` double,
  `duration_days` double,
  `at_search_edge` boolean,
  `bls_power` double,
  `depth_ppm` double,
  `noise_ppm` double,
  `n_in_transit` double,
  `n_transits` double,
  `snr` double,
  `st_tmag` double,
  `st_rad` double,
  `st_logg` double,
  `st_teff` double,
  `duty_cycle` double,
  `depth_odd_ppm` double,
  `depth_even_ppm` double,
  `odd_even_sigma` double,
  `secondary_depth_ppm` double,
  `secondary_sigma` double,
  `secondary_ratio` double,
  `shape_ratio` double,
  `red_noise_beta` double,
  `snr_red` double,
  `depth_chi2` double,
  `max_transit_frac` double,
  `variability_ppm` doubl

## Setting it up in Athena

In the Athena query editor (region **us-east-2**, query results saved to `s3://exoplanet-pipeline-emaad/athena-results/`):

```sql
CREATE DATABASE exoplanet;
-- then run the CREATE EXTERNAL TABLE statement printed above
MSCK REPAIR TABLE exoplanet.pipeline_results;   -- registers the run=... partitions
```

## Queries

**Counts by label and status** (scanned just **0.28 KB**, because Parquet stores data by column and the query reads only three):

```sql
SELECT run, label, status, COUNT(*) AS stars
FROM exoplanet.pipeline_results
GROUP BY run, label, status
ORDER BY label, status;
```

**Recovery rate across the full training set**, the same "match at SNR ≥ 7" rule as `evaluate()`:

```sql
SELECT label,
       COUNT(*) AS stars,
       SUM(CASE WHEN abs(period_days / catalog_period - 1) < 0.01
                 AND snr >= 7 THEN 1 ELSE 0 END) AS recovered,
       ROUND(100.0 * SUM(CASE WHEN abs(period_days / catalog_period - 1) < 0.01
                               AND snr >= 7 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct
FROM exoplanet.pipeline_results
WHERE run = 'train_v1' AND status = 'ok'
GROUP BY label;
```

| Label | Stars | Recovered | % |
|---|---|---|---|
| planet | 419 | 376 | **89.7%** |
| false_positive | 518 | 382 | 73.7% |

**89.7% of 419 confirmed planets**, matching the 90% from the 50-planet benchmark in notebook 03, but on eight times as many planets.

---
# Summary

- **Cloud run:** 1,066 stars, 32 parallel jobs, zero failures, 4.5 hours vs 11 locally, about $0.30–0.40
- **Bottleneck diagnosed:** account concurrency quota, load imbalance from target ordering, slower cloud CPUs, each with a fix
- **Reproducibility:** 933 of 933 verdicts identical to the laptop run; 99% of measurements agree to within 0.01%
- **Data lake:** results stored as partitioned Parquet in S3, queryable with SQL in Athena
- **Headline result:** the pipeline recovers **~90% of 419 confirmed planets**